In [1]:
from langgraph.graph import StateGraph, START , END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
import os

In [2]:
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
# Agent 1 - GPT-4o Mini (via OpenRouter)
openai_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# Agent 2 - Llama 3 (via OpenRouter)
huggingface_llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    temperature=0.7,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# Agent 3 - Qwen Coder (via OpenRouter)
groq_llm = ChatOpenAI(
    model="qwen/qwen3-coder",
    temperature=0.2,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
     max_tokens=1000
)

In [5]:
class ConditionalState(TypedDict, total=False):
    user_request: str
    route: str
    final_response: str

In [6]:
## This node represents the initial user request step. It receives the state, prints a status message, and passes the same state forward to the router
def user_request(state: ConditionalState):
    print("User request received")
    return state

In [7]:
def router_agent(state: ConditionalState):
    print("Router agent started: OpenAI routing agent")
    response = openai_llm.invoke(f"""
    You are a request routing agent.

    TASK:
    Classify the user request into exactly one route.

    USER REQUEST:
    {state['user_request']}

    ROUTES:
    - chat: simple conversation, greetings, explanations, or general questions
    - code: programming, debugging, writing code, or software development tasks
    - writer: current events, online research, web content, articles, or writing based on external information

    STRICT OUTPUT RULE:
    Return only one word: chat, code, or writer.
    """)

    route = response.content.strip().lower()

    if route not in {"chat", "code", "writer"}:
        route = "chat"

    print(f"Router selected route: {route}")
    return {"route": route}

In [8]:
## This function is used by LangGraph's conditional edges. It reads the route chosen by the router and returns it so the graph can choose the next node.
def route_request(state: ConditionalState):
    return state["route"]

In [9]:
def simple_chat_agent(state: ConditionalState):
    print("Simple chat agent started: OpenAI chat agent")
    response = openai_llm.invoke(f"""
    You are a helpful and friendly chat assistant.

    USER REQUEST:
    {state['user_request']}

    TASK:
    Answer clearly and naturally.
    """)

    print("Simple chat agent completed")
    return {"final_response": response.content.strip()}

In [10]:
def code_agent(state: ConditionalState):
    print("Code agent started: Groq coding agent")
    response = groq_llm.invoke(f"""
    You are a senior software engineering assistant.

    USER REQUEST:
    {state['user_request']}

    TASK:
    Provide a practical coding answer. Include code examples when useful.
    Be clear, correct, and concise.
    """)

    print("Code agent completed")
    return {"final_response": response.content.strip()}

In [11]:
def writer_agent(state: ConditionalState):
    print("Writer agent started: Hugging Face research/writer agent")
    response = huggingface_llm.invoke(f"""
    You are a professional research and content writing assistant.

    USER REQUEST:
    {state['user_request']}

    TASK:
    Write a clear, well-structured response for the user's information or writer-style request.

    IMPORTANT:
    If the request needs live web data, say that live browsing is not available in this workflow
    and provide a general answer based on the given request.
    """)

    print("Writer agent completed")
    return {"final_response": response.content.strip()}

In [12]:
graph = StateGraph(ConditionalState)
graph.add_node("user_request", user_request)
graph.add_node("router_agent", router_agent)
graph.add_node("simple_chat_agent", simple_chat_agent)
graph.add_node("code_agent", code_agent)
graph.add_node("writer_agent", writer_agent)

In [14]:
graph.add_edge(START, "user_request")
graph.add_edge("user_request", "router_agent")
graph.add_conditional_edges("router_agent", route_request, {
    "chat": "simple_chat_agent",
    "code": "code_agent",
    "writer": "writer_agent"
})
graph.add_edge("simple_chat_agent", END)
graph.add_edge("code_agent", END)
graph.add_edge("writer_agent", END)

In [15]:
app =graph.compile()

In [18]:
user_query = "write a email to teamlead about the project update and next steps"
result = app.invoke({"user_request": user_query})

User request received
Router agent started: OpenAI routing agent
Router selected route: writer
Writer agent started: Hugging Face research/writer agent
Writer agent completed


In [19]:
print("Final response from the selected agent:")
print(result["final_response"])

Final response from the selected agent:
Here is a clear and well-structured email to the team lead about the project update and next steps:

Subject: Project Update and Next Steps

Dear [Team Lead's Name],

I hope this email finds you well. I am writing to provide you with an update on the current status of our project and outline the next steps that we will take to move forward.

As of today, we have completed [mention the tasks or milestones that have been achieved]. The team has worked diligently to ensure that all aspects of the project are progressing as planned, and we are on track to meet our deadlines.

Below are the key highlights of the project's current status:

* [Mention the progress made so far, including any notable achievements or successes]
* [Highlight any challenges or obstacles that have been overcome]
* [Provide an overview of the project's financials, if applicable]

Moving forward, our next steps will be to:

* [Outline the specific tasks or activities that need 

In [ ]:
print(app.get_graph().draw_mermaid())